In [1]:
from brainhack import Signal, Tukey, System, Sequence, Simulator, Duration, Angle, Frequency, CompositeDictionary, GridRuns
import numpy as np
import matplotlib.pyplot as plt

# Helper function for `Applications` and later sections
def print_signals(mapping: CompositeDictionary):
    # Printing the mapping for every computed signal
    print('Shape:', mapping[list(mapping.keys())[0]].shape)
    for key in mapping.keys():
        print(str(key.name).rjust(13), mapping[key].tolist(), sep=' = ')

In [3]:
pulse = Tukey(
    shape     = .0,
    duration  = Duration.from_milli(1),   # in s
    flipAngle = Angle.from_degrees(180),  # in degree
    offset    = Frequency.from_kilo(7),   # in Hz
)

sequence = Sequence(
    signal            = Signal.ALL,
    pulse             = pulse,
    N_pulsePerOffset  = 1,
    N_pulse           = 6,
    N_burst           = 10,
    N_adc             = 96,
    N_dummyADC        = 0,
    dt_interPulse     = Duration.from_milli(1.5),   # in s
    TR_burst          = Duration.from_milli(100),   # in s
    dt_lastBurst      = Duration.from_milli(9),     # in s
    es                = Duration.from_milli(6),     # in s
    tr                = Duration.from_seconds(20),  # in s
    readout_flipAngle = Angle.from_degrees(6),      # in degrees
)

# system = System(
#     pulse                        = pulse,
#     poolFree_M0                  = 1,
#     poolFree_T1                  = Duration.from_milli(1000),     # in s
#     poolFree_T2                  = Duration.from_milli(100),      # in s
#     poolFreeBound_exchangeRate   = Frequency.from_hertz(20),      # in s^-1
#     poolBound_M0                 = 0.1,
#     poolBound_T1                 = Duration.from_milli(1000),     # in s 
#     poolBound_T1D                = Duration.from_milli(10),       # in s
#     poolBound_T2                 = Duration.from_micro(10),       # in s
#     poolBound_lineshapeAsymmetry = Frequency.from_hertz(0)  # in s^-1
# )

system = System(
    pulse                        = pulse,
    poolFree_M0                  = [1],
    poolFree_T1                  = [1],                 # in s
    poolFree_T2                  = [100e-3],            # in s
    poolFreeBound_exchangeRate   = [[20, 20]],          # in s^-1
    poolBound_M0                 = [0.1, 0.1],
    poolBound_T1                 = [1, 1],              # in s 
    poolBound_T1D                = [10e-3, 5e-3],       # in s
    poolBound_T2                 = [10e-6, 10e-6],      # in s
    poolBound_lineshapeAsymmetry = [0, 0],              # in s^-1
)

simulator = Simulator(
    system             = system,
    sequence           = sequence,
    output_vectorSlice = slice(1),
    export_readMatrix  = True,
)

sim = simulator.copy()
sim.output_vectorSlice = slice(None)
sim.export_readMatrix = True
data = GridRuns(
    sim,
    ['flipAngle'],
    {
        'flipAngle': (simulator.pulse.flipAngle * np.linspace(1, 1, 1)).astype(Angle)
    },
    safe=True,
)

# print('Readout matrix shape:', data['readout'].shape)
# print_signals(CompositeDictionary(data))

out = CompositeDictionary(data)
print_signals(out[Signal.ALL])

Shape: (6, 1)
      MTd_ALT = [[0.6477916515324613], [0.03459965724859253], [-1.14089727171715e-07], [0.034759006134468654], [-1.1721632197171501e-07], [1.0]]
       MTd_CM = [[0.6422111772869529], [0.03378439075583961], [0.0], [0.03378439075583961], [0.0], [1.0]]
 MTs_Negative = [[0.7517917238576683], [0.05452591961331791], [8.33004684286396e-07], [0.05167743181599185], [6.10922474814797e-07], [1.0]]
 MTs_Positive = [[0.7517917238576683], [0.05452591961331791], [8.33004684286396e-07], [0.05167743181599185], [6.10922474814797e-07], [1.0]]
          MT0 = [[0.9999999989959113], [0.09999999989959106], [0.0], [0.09999999989959106], [0.0], [1.0]]
          MTs = [[0.7517917238576683], [0.05452591961331791], [8.33004684286396e-07], [0.05167743181599185], [6.10922474814797e-07], [1.0]]
      ihMT_CM = [[0.21916109314143095], [0.0414830577149566], [1.666009368572792e-06], [0.035786082120304485], [1.221844949629594e-06], [0.0]]
     ihMT_ALT = [[0.20800014465041405], [0.039852524729450764], [1